# Pre-processing and data load

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import glob, os

In [3]:
#from https://cprimozic.net/notes/posts/machine-learning-benchmarks-on-the-7900-xtx/

gpus = tf.config.list_physical_devices('GPU')
print(gpus)
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
  except RuntimeError as e:
    print(e)

[]


In [4]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())


[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 931268246119189513
xla_global_id: -1
]


In [53]:
path = "/content/drive/MyDrive/TrackML_Data/train_3_events/"
# cells, hits, particles, truth = [], [], [], []

data = pd.read_csv(path + "event000001000-cells.csv")

df_temp = pd.read_csv(path + "event000001000-hits.csv")
data = pd.merge(data, df_temp, on='hit_id', how='inner')

df_temp = pd.read_csv(path + "event000001000-truth.csv")
data = pd.merge(data, df_temp, on='hit_id', how='inner')

df_temp = pd.read_csv(path + "event000001000-particles.csv")
data = pd.merge(data, df_temp, on='particle_id', how='inner')

print(data.head())


# data.join(df_temp.set_index('hit_id'), on='hit_id')
# print(data.head())

# data.join(df_temp.set_index('particle_id'), on='particle_id')
# # truth.append(df_temp)
# print(data.head())

# for filename in os.listdir(path):
#     print(f"Loading {filename}")
#     df_temp = pd.read_csv(path + filename)
#     if filename.endswith("cells.csv"):
#         cells.append(df_temp)
#         data.merge(pd.concat(cells))
#     if filename.endswith("hits.csv"):
#         hits.append(df_temp)
#         data.merge(pd.concat(hits), on = "hit_id")
#     if filename.endswith("particles.csv"):
#         particles.append(df_temp)
#         data.merge(pd.concat(particles), on = "particle_id")
#     if filename.endswith("truth.csv"):
#         truth.append(df_temp)
#         data.merge(pd.concat(truth), on = "hit_id")



   hit_id  ch0   ch1     value        x         y       z  volume_id  \
0       2   68   446  0.334087 -55.3361  0.635342 -1502.5          7   
1       4  181  1181  0.323907 -96.1091 -8.241030 -1502.5          7   
2       5  256   590  0.296566 -62.6736 -9.371200 -1502.5          7   
3       6  241   489  0.289269 -57.0687 -8.177770 -1502.5          7   
4       7  103   779  0.304021 -73.8723 -2.578900 -1502.5          7   

   layer_id  module_id  ...       tpz    weight        vx        vy       vz  \
0         2          1  ... -15.49220  0.000010 -0.015802  0.006381  1.16279   
1         2          1  ...  -3.70232  0.000008 -0.000486 -0.015051  5.75865   
2         2          1  ...  -6.57318  0.000009  0.018365 -0.016865  4.19268   
3         2          1  ... -10.46690  0.000008  0.010383 -0.012398  3.92894   
4         2          1  ...  -9.13010  0.000007 -0.004178  0.004751 -5.12884   

         px        py        pz  q  nhits  
0 -0.569670 -0.011187 -15.49600  1     10 

# Data splitting and model building

In [54]:
data.columns

Index(['hit_id', 'ch0', 'ch1', 'value', 'x', 'y', 'z', 'volume_id', 'layer_id',
       'module_id', 'particle_id', 'tx', 'ty', 'tz', 'tpx', 'tpy', 'tpz',
       'weight', 'vx', 'vy', 'vz', 'px', 'py', 'pz', 'q', 'nhits'],
      dtype='object')

In [55]:
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import tensorflow as tf

x = data.drop(columns='particle_id')
y = data['particle_id'].values

print(x[:5], y[:5])

x, y = shuffle(x, y)

print(x[:5], y[:5])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)


   hit_id  ch0   ch1     value        x         y       z  volume_id  \
0       2   68   446  0.334087 -55.3361  0.635342 -1502.5          7   
1       4  181  1181  0.323907 -96.1091 -8.241030 -1502.5          7   
2       5  256   590  0.296566 -62.6736 -9.371200 -1502.5          7   
3       6  241   489  0.289269 -57.0687 -8.177770 -1502.5          7   
4       7  103   779  0.304021 -73.8723 -2.578900 -1502.5          7   

   layer_id  module_id  ...       tpz    weight        vx        vy       vz  \
0         2          1  ... -15.49220  0.000010 -0.015802  0.006381  1.16279   
1         2          1  ...  -3.70232  0.000008 -0.000486 -0.015051  5.75865   
2         2          1  ...  -6.57318  0.000009  0.018365 -0.016865  4.19268   
3         2          1  ... -10.46690  0.000008  0.010383 -0.012398  3.92894   
4         2          1  ...  -9.13010  0.000007 -0.004178  0.004751 -5.12884   

         px        py        pz  q  nhits  
0 -0.569670 -0.011187 -15.49600  1     10 

In [ ]:
model = models.Sequential([
    layers.Dense(26, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(9, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
